In [1]:
import kagglehub
path = kagglehub.dataset_download("shriyashjagtap/e-commerce-customer-for-behavior-analysis")

100%|██████████| 9.94M/9.94M [00:00<00:00, 37.2MB/s]

Extracting files...


In [4]:
import os
import pandas as pd

# List files in the downloaded directory to find the dataset file
dataset_path = path # 'path' variable is already defined from previous execution
print(f"Listing files in: {dataset_path}")
for root, dirs, files in os.walk(dataset_path):
    for file in files:
        print(os.path.join(root, file))

# Corrected filename based on the output of the previous execution
data_file = os.path.join(dataset_path, 'ecommerce_customer_data_large.csv') # Changed from 'Ecommerce_Customer_Behavior.csv'

# Load the dataset into a pandas DataFrame
try:
    df = pd.read_csv(data_file)
    print("\nDataset loaded successfully!")
except FileNotFoundError:
    print(f"Error: '{data_file}' not found. Please check the file name from the list above.")
    df = None


Listing files in: /root/.cache/kagglehub/datasets/shriyashjagtap/e-commerce-customer-for-behavior-analysis/versions/4
/root/.cache/kagglehub/datasets/shriyashjagtap/e-commerce-customer-for-behavior-analysis/versions/4/ecommerce_customer_data_custom_ratios.csv
/root/.cache/kagglehub/datasets/shriyashjagtap/e-commerce-customer-for-behavior-analysis/versions/4/ecommerce_customer_data_large.csv

Dataset loaded successfully!


Now that the data is loaded (or an attempt has been made to load it), let's inspect its first few rows and summary information.

In [3]:
if df is not None:
    print("\nFirst 5 rows of the dataset:")
    display(df.head())
    print("\nDataset Info:")
    display(df.info())


### Data Preprocessing

Before building a Deep Learning model, we need to preprocess the data. This typically involves:
1.  **Handling Missing Values**: Identify and decide on strategies to fill or remove missing data.
2.  **Data Type Conversion**: Ensure columns are in appropriate data types (e.g., converting 'Purchase Date' to datetime objects).
3.  **Feature Engineering**: Create new features that might be useful for the model.
4.  **Encoding Categorical Variables**: Convert categorical text data into numerical format that a DL model can understand.
5.  **Scaling Numerical Features**: Normalize or standardize numerical features to a similar range.
6.  **Splitting Data**: Divide the dataset into training, validation, and test sets.


In [6]:
# Check for missing values
print("\nMissing values per column:")
display(df.isnull().sum())

# Convert 'Purchase Date' to datetime objects (already done, but keeping for completeness if cell is run independently)
df['Purchase Date'] = pd.to_datetime(df['Purchase Date'])

# Fill missing 'Returns' values with 0.0 (assuming NaN means no return)
df['Returns'] = df['Returns'].fillna(0.0)

print("\nData types after converting 'Purchase Date' and handling 'Returns' NaNs:")
display(df.info())



Missing values per column:


,0
Customer ID,0
Purchase Date,0
Product Category,0
Product Price,0
Quantity,0
Total Purchase Amount,0
Payment Method,0
Customer Age,0
Returns,47382
Customer Name,0



Data types after converting 'Purchase Date' and handling 'Returns' NaNs:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 13 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   Customer ID            250000 non-null  int64         
 1   Purchase Date          250000 non-null  datetime64[ns]
 2   Product Category       250000 non-null  object        
 3   Product Price          250000 non-null  int64         
 4   Quantity               250000 non-null  int64         
 5   Total Purchase Amount  250000 non-null  int64         
 6   Payment Method         250000 non-null  object        
 7   Customer Age           250000 non-null  int64         
 8   Returns                250000 non-null  float64       
 9   Customer Name          250000 non-null  object        
 10  Age                    250000 non-null  int64         
 11  Gender                 250000 

None

In [7]:
# Drop irrelevant columns
df = df.drop(['Customer ID', 'Customer Name'], axis=1)

# One-hot encode categorical features
df = pd.get_dummies(df, columns=['Product Category', 'Payment Method', 'Gender'], drop_first=True)

print("\nDataFrame after dropping IDs and one-hot encoding categorical features:")
display(df.head())
display(df.info())



DataFrame after dropping IDs and one-hot encoding categorical features:


,Purchase Date,Product Price,Quantity,Total Purchase Amount,Customer Age,Returns,Age,Churn,Product Category_Clothing,Product Category_Electronics,Product Category_Home,Payment Method_Credit Card,Payment Method_PayPal,Gender_Male
0,2023-05-03 21:30:02,177,1,2427,31,1.0,31,0,False,False,True,False,True,False
1,2021-05-16 13:57:44,174,3,2448,31,1.0,31,0,False,True,False,False,True,False
2,2020-07-13 06:16:57,413,1,2345,31,1.0,31,0,False,False,False,True,False,False
3,2023-01-17 13:14:36,396,3,937,31,0.0,31,0,False,True,False,False,False,False
4,2021-05-01 11:29:27,259,4,2598,31,1.0,31,0,False,False,False,False,True,False


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 14 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   Purchase Date                 250000 non-null  datetime64[ns]
 1   Product Price                 250000 non-null  int64         
 2   Quantity                      250000 non-null  int64         
 3   Total Purchase Amount         250000 non-null  int64         
 4   Customer Age                  250000 non-null  int64         
 5   Returns                       250000 non-null  float64       
 6   Age                           250000 non-null  int64         
 7   Churn                         250000 non-null  int64         
 8   Product Category_Clothing     250000 non-null  bool          
 9   Product Category_Electronics  250000 non-null  bool          
 10  Product Category_Home         250000 non-null  bool          
 11  Payment Metho

None

From the missing values check, we can see that 'Returns' has a significant number of missing values. We will address these along with other preprocessing steps for categorical and numerical features.

### Further Feature Engineering and Scaling

Let's extract more useful features from `Purchase Date` and then scale our numerical features. Finally, we'll split the data into training and testing sets.

In [8]:
# Extract features from 'Purchase Date'
df['Purchase_Month'] = df['Purchase Date'].dt.month
df['Purchase_DayOfWeek'] = df['Purchase Date'].dt.dayofweek
df['Purchase_Day'] = df['Purchase Date'].dt.day
df['Purchase_Year'] = df['Purchase Date'].dt.year

# Drop the original 'Purchase Date' column as it's no longer needed
df = df.drop('Purchase Date', axis=1)

print("\nDataFrame after extracting date features:")
display(df.head())



DataFrame after extracting date features:


,Product Price,Quantity,Total Purchase Amount,Customer Age,Returns,Age,Churn,Product Category_Clothing,Product Category_Electronics,Product Category_Home,Payment Method_Credit Card,Payment Method_PayPal,Gender_Male,Purchase_Month,Purchase_DayOfWeek,Purchase_Day,Purchase_Year
0,177,1,2427,31,1.0,31,0,False,False,True,False,True,False,5,2,3,2023
1,174,3,2448,31,1.0,31,0,False,True,False,False,True,False,5,6,16,2021
2,413,1,2345,31,1.0,31,0,False,False,False,True,False,False,7,0,13,2020
3,396,3,937,31,0.0,31,0,False,True,False,False,False,False,1,1,17,2023
4,259,4,2598,31,1.0,31,0,False,False,False,False,True,False,5,5,1,2021


In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
X = df.drop('Churn', axis=1)
y = df['Churn']

# Identify numerical columns for scaling (exclude boolean and date features that are already numerical but not raw values)
numerical_cols = ['Product Price', 'Quantity', 'Total Purchase Amount', 'Customer Age', 'Returns', 'Age', 'Purchase_Month', 'Purchase_DayOfWeek', 'Purchase_Day', 'Purchase_Year']

# Initialize StandardScaler
scaler = StandardScaler()

# Apply scaling to numerical columns
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

print("\nFeatures (X) after scaling numerical columns:")
display(X.head())

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nShape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")



Features (X) after scaling numerical columns:


,Product Price,Quantity,Total Purchase Amount,Customer Age,Returns,Age,Product Category_Clothing,Product Category_Electronics,Product Category_Home,Payment Method_Credit Card,Payment Method_PayPal,Gender_Male,Purchase_Month,Purchase_DayOfWeek,Purchase_Day,Purchase_Year
0,-0.548497,-1.417183,-0.206842,-0.832956,1.209809,-0.832956,False,False,True,False,True,False,-0.354994,-0.500295,-1.436999,1.510049
1,-0.569663,-0.003489,-0.192285,-0.832956,1.209809,-0.832956,False,True,False,False,True,False,-0.354994,1.502639,0.039198,-0.350919
2,1.116549,-1.417183,-0.263685,-0.832956,1.209809,-0.832956,False,False,False,True,False,False,0.241446,-1.501762,-0.301463,-1.281403
3,0.996610,-0.003489,-1.239719,-0.832956,-0.826577,-0.832956,False,True,False,False,False,False,-1.547873,-1.001028,0.152752,1.510049
4,0.030036,0.703358,-0.088304,-0.832956,1.209809,-0.832956,False,False,False,False,True,False,-0.354994,1.001905,-1.664106,-0.350919



Shape of X_train: (200000, 16)
Shape of X_test: (50000, 16)
Shape of y_train: (200000,)
Shape of y_test: (50000,)


### Building and Training the Deep Learning Model

Now that our data is prepared, let's define and train a Deep Learning model using TensorFlow/Keras. We'll start with a simple feed-forward neural network.

In [10]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Get the number of features for the input layer
input_dim = X_train.shape[1]

# Define the model architecture
model = Sequential([
    Dense(128, activation='relu', input_shape=(input_dim,)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid') # Output layer for binary classification
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Display model summary
print("\nModel Architecture:")
model.summary()



Model Architecture:


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         2,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,545 (49.00 KB)

 Trainable params: 12,545 (49.00 KB)

 Non-trainable params: 0 (0.00 B)

The model is now defined and compiled. Let's train it using our training data. We'll also use a validation split to monitor performance during training.

In [11]:
# Train the model
history = model.fit(
    X_train,
    y_train,
    epochs=20, # Number of training epochs
    batch_size=32, # Batch size for training
    validation_split=0.2, # Use 20% of training data for validation
    verbose=1 # Show progress bar during training
)

print("\nModel training complete!")


Epoch 1/20
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - accuracy: 0.7988 - loss: 0.5080 - val_accuracy: 0.8006 - val_loss: 0.4998
Epoch 2/20
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.7992 - loss: 0.5028 - val_accuracy: 0.8006 - val_loss: 0.4997
Epoch 3/20
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.7992 - loss: 0.5025 - val_accuracy: 0.8006 - val_loss: 0.4997
Epoch 4/20
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.7992 - loss: 0.5019 - val_accuracy: 0.8006 - val_loss: 0.4997
Epoch 5/20
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - accuracy: 0.7992 - loss: 0.5019 - val_accuracy: 0.8006 - val_loss: 0.4996
Epoch 6/20
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - accuracy: 0.7992 - loss: 0.5018 - val_accuracy: 0.8006 - val_loss: 0.4999
Epoch 7/20
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.7992 - loss: 0.5017 - val_accuracy: 0.8006 - val_loss: 0.5000
Epoch 8/20
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - accuracy: 0.7992 - loss: 0.5

The model has been trained. Next, we will evaluate its performance on the unseen test data.

In [12]:
# Evaluate the model on the test data
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

print(f"\nTest Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")



Test Loss: 0.5012
Test Accuracy: 0.7995


The model's performance on the test set is now available. We can further analyze the model's predictions and performance using metrics like precision, recall, F1-score, and a confusion matrix if needed.

### Saving the Model and Scaler

Before we build the Streamlit application, we need to save our trained Keras model and the `StandardScaler` object. This way, the Streamlit app can load them without needing to retrain the model or refit the scaler every time.

In [13]:
import joblib

# Save the Keras model
model.save('churn_prediction_model.keras')
print("Keras model saved as 'churn_prediction_model.keras'")

# Save the StandardScaler
joblib.dump(scaler, 'scaler.pkl')
print("StandardScaler saved as 'scaler.pkl'")


Keras model saved as 'churn_prediction_model.keras'
StandardScaler saved as 'scaler.pkl'


### Streamlit Application

Now, let's create the Python script for the Streamlit application. This script will:
1.  Load the saved model and scaler.
2.  Create a user interface with input fields for each feature.
3.  Preprocess the user input.
4.  Make a churn prediction.
5.  Display the prediction and potentially some visuals.

Copy the code below into a file named `app.py` on your local machine.

In [14]:
%%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import joblib
import datetime
import plotly.express as px

# Load the trained model and scaler
@st.cache_resource
def load_artifacts():
    model = load_model('churn_prediction_model.keras')
    scaler = joblib.load('scaler.pkl')
    return model, scaler

model, scaler = load_artifacts()

# Get feature names from the original training data (assuming X_train was a DataFrame)
# We need this to ensure the input order is correct for prediction
# This assumes X is available in the environment from previous cells
# In a real app, you might hardcode this or save X.columns as well
# For simplicity, let's derive it from the last state of X from the notebook.
# In a production app, ensure these are explicitly defined or loaded.

# We need the original column order from X *before* scaling for feature names mapping
# Let's recreate a dummy X to get column names if not explicitly saved
# (Ideally, you would save X.columns during training phase)
# For this demonstration, we'll manually list them based on the notebook's preprocessing
original_feature_columns = ['Product Price', 'Quantity', 'Total Purchase Amount', 'Customer Age', 'Returns', 'Age',
                            'Product Category_Clothing', 'Product Category_Electronics', 'Product Category_Home',
                            'Payment Method_Credit Card', 'Payment Method_PayPal', 'Gender_Male',
                            'Purchase_Month', 'Purchase_DayOfWeek', 'Purchase_Day', 'Purchase_Year']


st.set_page_config(page_title="Customer Churn Prediction", layout="wide")

st.title("🛍️ E-commerce Customer Churn Prediction")
st.write("This application predicts whether an e-commerce customer is likely to churn based on their behavior and demographics. Enter the customer's details below and click 'Predict Churn' to see the results!")

st.sidebar.header("About the Model")
st.sidebar.info("This model is a Deep Learning neural network trained on e-commerce customer data. It uses various features like purchase history, demographics, and payment methods to predict customer churn.")

# Input features from user
st.header("Customer Information")

with st.expander("Demographics and Purchase Details", expanded=True):
    col1, col2, col3 = st.columns(3)
    with col1:
        customer_age = st.slider("Customer Age", min_value=18, max_value=80, value=30)
        age = st.slider("Age (from dataset, usually same as Customer Age)", min_value=18, max_value=80, value=30)
        gender = st.selectbox("Gender", ['Female', 'Male'], index=0)
    with col2:
        product_price = st.number_input("Product Price ($)", min_value=1, max_value=1000, value=150)
        quantity = st.number_input("Quantity", min_value=1, max_value=10, value=2)
        total_purchase_amount = st.number_input("Total Purchase Amount ($)", min_value=1, max_value=5000, value=300)
    with col3:
        payment_method = st.selectbox("Payment Method", ['Cash', 'Credit Card', 'PayPal'], index=0)
        returns = st.selectbox("Has Made Returns?", [0.0, 1.0], index=0)
        purchase_date = st.date_input("Last Purchase Date", datetime.date(2023, 1, 1))

# Preprocess input data
def preprocess_input(customer_age, age, gender, product_price, quantity, total_purchase_amount, payment_method, returns, purchase_date):
    input_data = {
        'Product Price': product_price,
        'Quantity': quantity,
        'Total Purchase Amount': total_purchase_amount,
        'Customer Age': customer_age,
        'Returns': returns,
        'Age': age,
        'Product Category_Clothing': False, # Default, will be updated if we add product category input
        'Product Category_Electronics': False,
        'Product Category_Home': False,
        'Payment Method_Credit Card': False,
        'Payment Method_PayPal': False,
        'Gender_Male': False,
        'Purchase_Month': purchase_date.month,
        'Purchase_DayOfWeek': purchase_date.weekday(),
        'Purchase_Day': purchase_date.day,
        'Purchase_Year': purchase_date.year
    }

    # Handle Payment Method
    if payment_method == 'Credit Card':
        input_data['Payment Method_Credit Card'] = True
    elif payment_method == 'PayPal':
        input_data['Payment Method_PayPal'] = True

    # Handle Gender
    if gender == 'Male':
        input_data['Gender_Male'] = True

    # Convert to DataFrame in the exact order as original_feature_columns for scaling
    input_df = pd.DataFrame([input_data], columns=original_feature_columns)

    # Scale numerical features (ensure numerical_cols from training match)
    numerical_cols_app = ['Product Price', 'Quantity', 'Total Purchase Amount', 'Customer Age', 'Returns', 'Age', 'Purchase_Month', 'Purchase_DayOfWeek', 'Purchase_Day', 'Purchase_Year']
    input_df[numerical_cols_app] = scaler.transform(input_df[numerical_cols_app])

    return input_df


if st.button("Predict Churn"):
    processed_input = preprocess_input(
        customer_age, age, gender, product_price, quantity, total_purchase_amount, payment_method, returns, purchase_date
    )

    prediction_proba = model.predict(processed_input)[0][0]

    st.subheader("Prediction Result:")
    if prediction_proba >= 0.5:
        st.error(f"Customer is LIKELY to Churn! (Probability: {prediction_proba:.2f})")
    else:
        st.success(f"Customer is UNLIKELY to Churn. (Probability: {prediction_proba:.2f})")

    st.write("--- ")
    st.subheader("Prediction Visualisation")
    chart_data = pd.DataFrame({
        'Category': ['Churn Probability', 'No Churn Probability'],
        'Value': [prediction_proba, 1 - prediction_proba]
    })
    fig = px.bar(chart_data, x='Category', y='Value',
                 color='Category',
                 color_discrete_map={'Churn Probability': 'red', 'No Churn Probability': 'green'},
                 title='Churn Probability Breakdown')
    fig.update_layout(yaxis_range=[0,1])
    st.plotly_chart(fig, use_container_width=True)


st.markdown("---")
st.write("Created for E-commerce Customer Behavior Analysis")


Writing app.py


### How to Run the Streamlit Application

1.  **Save the `app.py` file:** Ensure you have copied the content of the `%%writefile app.py` cell into a file named `app.py` in the same directory where you saved `churn_prediction_model.keras` and `scaler.pkl`.

2.  **Install Streamlit:** If you don't have Streamlit installed, open your terminal or command prompt and run:
    ```bash
    pip install streamlit pandas numpy tensorflow scikit-learn plotly
    ```

3.  **Run the application:** In your terminal or command prompt, navigate to the directory where you saved `app.py` and run:
    ```bash
    streamlit run app.py
    ```

    This will open a new tab in your web browser with the Streamlit application.

In [15]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 50.2 MB/s eta 0:00:00


In [17]:
from pyngrok import ngrok
ngrok.set_auth_token("3H2nxZtP4iC5L9tX9K97OPLut9W_4JsZrRVF5aRFQpQCCePy1")

In [18]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://vanity-amperage-commodore.ngrok-free.dev" -> "http://localhost:8501"


In [19]:

!streamlit run app.py &


2026-07-27 13:36:13.991 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.221.37.115:8501

2026-07-27 13:36:24.196840: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
2026-07-27 13:37:09.868 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
  Stopping...
